# after normalisation, synthetic data generation

In [ ]:
# WGAN PATCH EXTRACTOR v4 — STREAMING TO DISK + FLOOD EVENT DEDUPLICATION
# ============================================================================
# Fixes two crash causes from v3:
#   1. 174,000+ patches accumulated in RAM (~348GB) → crash
#      Fix: write patches to disk immediately, never accumulate in memory
#   2. Consecutive flood hours from the same storm produce near-identical
#      patches (1,590 flood events in first 2,000 hours vs expected ~175)
#      Fix: minimum 6-hour gap between extracted patches (one per storm
#      period, not one per hour of a multi-day flood event)
#
# Output: patches saved as individual .npy files, then stacked at the end
# ============================================================================
import os
import time
import numpy as np
import pandas as pd
from collections import deque

CSV_PATH   = "merged_output_hourly_imerg/merged_data_normalised.csv"
OUTPUT_DIR = "wgan_patches"
TMP_DIR    = os.path.join(OUTPUT_DIR, "tmp_patches")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

N_LAT           = 140
N_LON           = 131
PATCH_SIZE      = 64
CELLS_PER_HOUR  = N_LAT * N_LON
N_HOURS         = 21888
CONTEXT_HOURS   = 6
FLOOD_THRESHOLD = 95.0
MIN_GAP_HOURS   = 6   # minimum hours between extracted flood patches
                       # prevents near-duplicate patches from same storm

header_cols  = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
TARGET_COL   = "flood_composite_6h_ahead"
FEATURE_COLS = [c for c in header_cols
                if c not in {"time", "latitude", "longitude"}]
N_FEATURES   = len(FEATURE_COLS)

print(f"Source: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")
print(f"Features: {N_FEATURES}, Patch: {PATCH_SIZE}x{PATCH_SIZE}")
print(f"Min gap between patches: {MIN_GAP_HOURS}h\n")


def extract_patch(grid_3d, lat_c, lon_c, size=PATCH_SIZE):
    half    = size // 2
    lat_min = max(0, lat_c - half)
    lat_max = min(grid_3d.shape[0], lat_c + half)
    lon_min = max(0, lon_c - half)
    lon_max = min(grid_3d.shape[1], lon_c + half)
    patch   = grid_3d[lat_min:lat_max, lon_min:lon_max, :]
    pad_top    = max(0, half - lat_c)
    pad_bottom = max(0, (lat_c + half) - grid_3d.shape[0])
    pad_left   = max(0, half - lon_c)
    pad_right  = max(0, (lon_c + half) - grid_3d.shape[1])
    if any([pad_top, pad_bottom, pad_left, pad_right]):
        patch = np.pad(patch,
                       ((pad_top, pad_bottom), (pad_left, pad_right), (0,0)),
                       mode="reflect")
    return patch


def augment(current, context):
    """8× augmentation: 4 rotations × 2 flips."""
    results = []
    for k in range(4):
        c   = np.rot90(current, k, axes=(0,1))
        ctx = np.stack([np.rot90(context[t], k, axes=(0,1))
                        for t in range(context.shape[0])])
        results.append((c.copy(), ctx.copy()))
        results.append((c[:, ::-1, :].copy(), ctx[:, :, ::-1, :].copy()))
    return results


print("Single-pass scan with rolling buffer + streaming to disk...")
buffer           = deque(maxlen=CONTEXT_HOURS + 1)
last_extract_hour = -MIN_GAP_HOURS  # allow extraction from hour 0
n_flood_seen     = 0
n_extracted      = 0
patch_index      = 0
scores_list      = []

t0 = time.time()

for hour_idx, chunk in enumerate(pd.read_csv(
        CSV_PATH, chunksize=CELLS_PER_HOUR, on_bad_lines="skip")):

    if len(chunk) != CELLS_PER_HOUR:
        continue

    arr  = chunk[FEATURE_COLS].to_numpy(dtype=np.float32)
    grid = arr.reshape(N_LAT, N_LON, N_FEATURES)
    buffer.append((hour_idx, grid))

    scores     = chunk[TARGET_COL].to_numpy()
    peak_score = np.nanmax(scores)

    if peak_score >= FLOOD_THRESHOLD:
        n_flood_seen += 1

        # Only extract if enough gap since last extraction
        if (hour_idx - last_extract_hour) >= MIN_GAP_HOURS:
            peak_cell    = int(np.nanargmax(scores))
            peak_lat_idx = peak_cell // N_LON
            peak_lon_idx = peak_cell % N_LON

            current_patch = extract_patch(grid, peak_lat_idx, peak_lon_idx)

            ctx_list  = list(buffer)[:-1]
            ctx_grids = []
            for _ in range(CONTEXT_HOURS - len(ctx_list)):
                ctx_grids.append(np.zeros((PATCH_SIZE, PATCH_SIZE, N_FEATURES),
                                           dtype=np.float32))
            for _, g in ctx_list[-CONTEXT_HOURS:]:
                ctx_grids.append(extract_patch(g, peak_lat_idx, peak_lon_idx))
            context_stack = np.stack(ctx_grids, axis=0)

            # Write each augmented patch directly to disk
            for aug_c, aug_ctx in augment(current_patch, context_stack):
                c_arr   = aug_c.transpose(2, 0, 1).astype(np.float32)
                ctx_arr = aug_ctx.transpose(0, 3, 1, 2).astype(np.float32)
                np.save(os.path.join(TMP_DIR, f"c_{patch_index:06d}.npy"),
                        c_arr)
                np.save(os.path.join(TMP_DIR, f"ctx_{patch_index:06d}.npy"),
                        ctx_arr)
                scores_list.append(float(peak_score))
                patch_index += 1

            n_extracted += 1
            last_extract_hour = hour_idx

    if (hour_idx + 1) % 2000 == 0:
        elapsed = time.time() - t0
        rate    = (hour_idx + 1) / elapsed
        eta     = (N_HOURS - hour_idx - 1) / rate
        print(f"  ... {hour_idx+1:,}/{N_HOURS} hrs | "
              f"flood hrs seen: {n_flood_seen} | "
              f"patches extracted: {n_extracted} | "
              f"{elapsed/60:.1f}min | ~{eta/60:.0f}min remaining")

elapsed = time.time() - t0
print(f"\n✓ Scan complete in {elapsed/60:.1f} min")
print(f"  Flood hours seen:     {n_flood_seen:,}")
print(f"  Patches extracted:    {n_extracted} "
      f"(after {MIN_GAP_HOURS}h deduplication)")
print(f"  After 8x augmentation: {patch_index}")

# Stack individual files into final arrays
print(f"\nStacking {patch_index} patches into final arrays...")
t0 = time.time()

all_current = np.stack([
    np.load(os.path.join(TMP_DIR, f"c_{i:06d}.npy"))
    for i in range(patch_index)
]).astype(np.float32)

all_context = np.stack([
    np.load(os.path.join(TMP_DIR, f"ctx_{i:06d}.npy"))
    for i in range(patch_index)
]).astype(np.float32)

all_scores = np.array(scores_list, dtype=np.float32)

print(f"  patches_current: {all_current.shape} "
      f"({all_current.nbytes/1e6:.0f} MB)")
print(f"  patches_context: {all_context.shape} "
      f"({all_context.nbytes/1e6:.0f} MB)")

np.save(os.path.join(OUTPUT_DIR, "patches_current.npy"), all_current)
np.save(os.path.join(OUTPUT_DIR, "patches_context.npy"), all_context)
np.save(os.path.join(OUTPUT_DIR, "patches_scores.npy"),  all_scores)

# Clean up tmp files
import shutil
shutil.rmtree(TMP_DIR)

total_mb = (all_current.nbytes + all_context.nbytes) / 1e6
print(f"\n✓ Saved to {OUTPUT_DIR}/ ({total_mb:.0f} MB)")
print(f"  Stacking time: {(time.time()-t0)/60:.1f} min")
print("  Next: run wgan_train.py")

Source: merged_output_hourly_imerg/merged_data_normalised.csv (79.65 GB)
Features: 21, Patch: 64x64
Min gap between patches: 6h

Single-pass scan with rolling buffer + streaming to disk...
  ... 2,000/21888 hrs | flood hrs seen: 1590 | patches extracted: 269 | 1.3min | ~13min remaining
  ... 4,000/21888 hrs | flood hrs seen: 2936 | patches extracted: 502 | 2.5min | ~11min remaining
  ... 6,000/21888 hrs | flood hrs seen: 4035 | patches extracted: 698 | 3.6min | ~9min remaining
  ... 8,000/21888 hrs | flood hrs seen: 5282 | patches extracted: 914 | 4.7min | ~8min remaining
  ... 10,000/21888 hrs | flood hrs seen: 6686 | patches extracted: 1155 | 5.9min | ~7min remaining
  ... 12,000/21888 hrs | flood hrs seen: 7521 | patches extracted: 1300 | 6.8min | ~6min remaining
  ... 14,000/21888 hrs | flood hrs seen: 8616 | patches extracted: 1494 | 7.9min | ~4min remaining
  ... 16,000/21888 hrs | flood hrs seen: 9804 | patches extracted: 1702 | 9.0min | ~3min remaining
  ... 18,000/21888 hrs | 

/var/folders/f9/b36b9whj67b0j3195q6474dh0000gn/T/ipykernel_16325/2249850365.py:97: RuntimeWarning: All-NaN slice encountered
  peak_score = np.nanmax(scores)
/var/folders/f9/b36b9whj67b0j3195q6474dh0000gn/T/ipykernel_16325/2249850365.py:97: RuntimeWarning: All-NaN slice encountered
  peak_score = np.nanmax(scores)
/var/folders/f9/b36b9whj67b0j3195q6474dh0000gn/T/ipykernel_16325/2249850365.py:97: RuntimeWarning: All-NaN slice encountered
  peak_score = np.nanmax(scores)
/var/folders/f9/b36b9whj67b0j3195q6474dh0000gn/T/ipykernel_16325/2249850365.py:97: RuntimeWarning: All-NaN slice encountered
  peak_score = np.nanmax(scores)
/var/folders/f9/b36b9whj67b0j3195q6474dh0000gn/T/ipykernel_16325/2249850365.py:97: RuntimeWarning: All-NaN slice encountered
  peak_score = np.nanmax(scores)
/var/folders/f9/b36b9whj67b0j3195q6474dh0000gn/T/ipykernel_16325/2249850365.py:97: RuntimeWarning: All-NaN slice encountered
  peak_score = np.nanmax(scores)



✓ Scan complete in 12.5 min
  Flood hours seen:     14,002
  Patches extracted:    2419 (after 6h deduplication)
  After 8x augmentation: 19352

Stacking 19352 patches into final arrays...


: 

In [12]:
# WGAN PATCH EXTRACTOR v5 — NO AUGMENTATION, 1H GAP, DISK STREAMING
# ============================================================================
# Changes from v4b:
#   1. MIN_GAP_HOURS reduced from 6 to 1 — extracts patches from every
#      consecutive flood hour, giving ~14,000 diverse patches instead of
#      ~175 unique storm events
#   2. Augmentation removed entirely — with 14,000 genuinely diverse patches
#      from different flood hours, augmentation is not needed and would
#      require 229GB disk space that is not available
#   3. Writes one patch per flood hour directly to tmp_patches/ as before
#
# Expected output: ~14,000 patches
#   patches_current.npy : ~4.8GB
#   patches_context.npy : ~29GB (after stacking)
# ============================================================================
import os
import time
import numpy as np
import pandas as pd
from collections import deque

CSV_PATH   = "merged_output_hourly_imerg/merged_data_normalised.csv"
OUTPUT_DIR = "wgan_patches"
TMP_DIR    = os.path.join(OUTPUT_DIR, "tmp_patches")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

N_LAT           = 140
N_LON           = 131
PATCH_SIZE      = 64
CELLS_PER_HOUR  = N_LAT * N_LON
N_HOURS         = 21888
CONTEXT_HOURS   = 6
FLOOD_THRESHOLD = 95.0
MIN_GAP_HOURS   = 1    # changed from 6 — every consecutive flood hour

header_cols  = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
TARGET_COL   = "flood_composite_6h_ahead"
FEATURE_COLS = [c for c in header_cols
                if c not in {"time", "latitude", "longitude"}]
N_FEATURES   = len(FEATURE_COLS)

print(f"Source: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")
print(f"Features: {N_FEATURES}, Patch: {PATCH_SIZE}x{PATCH_SIZE}")
print(f"Min gap between patches: {MIN_GAP_HOURS}h (no augmentation)\n")

# Check if tmp files already exist from previous run
existing = [f for f in os.listdir(TMP_DIR) if f.startswith("c_")]
if existing:
    print(f"Found {len(existing)} existing tmp files — skipping scan.")
    print("Run wgan_stack_patches.py to stack them.")
    raise SystemExit(0)


def extract_patch(grid_3d, lat_c, lon_c, size=PATCH_SIZE):
    half    = size // 2
    lat_min = max(0, lat_c - half)
    lat_max = min(grid_3d.shape[0], lat_c + half)
    lon_min = max(0, lon_c - half)
    lon_max = min(grid_3d.shape[1], lon_c + half)
    patch   = grid_3d[lat_min:lat_max, lon_min:lon_max, :]
    pad_top    = max(0, half - lat_c)
    pad_bottom = max(0, (lat_c + half) - grid_3d.shape[0])
    pad_left   = max(0, half - lon_c)
    pad_right  = max(0, (lon_c + half) - grid_3d.shape[1])
    if any([pad_top, pad_bottom, pad_left, pad_right]):
        patch = np.pad(patch,
                       ((pad_top, pad_bottom), (pad_left, pad_right), (0,0)),
                       mode="reflect")
    return patch


print("Single-pass scan — writing one patch per flood hour to disk...")
buffer            = deque(maxlen=CONTEXT_HOURS + 1)
last_extract_hour = -MIN_GAP_HOURS
n_flood_seen      = 0
n_extracted       = 0
patch_index       = 0
scores_list       = []
t0 = time.time()

for hour_idx, chunk in enumerate(pd.read_csv(
        CSV_PATH, chunksize=CELLS_PER_HOUR, on_bad_lines="skip")):

    if len(chunk) != CELLS_PER_HOUR:
        continue

    arr  = chunk[FEATURE_COLS].to_numpy(dtype=np.float32)
    grid = arr.reshape(N_LAT, N_LON, N_FEATURES)
    buffer.append((hour_idx, grid))

    scores     = chunk[TARGET_COL].to_numpy()
    valid      = scores[~np.isnan(scores)]
    peak_score = float(valid.max()) if len(valid) > 0 else 0.0

    if peak_score >= FLOOD_THRESHOLD:
        n_flood_seen += 1
        if (hour_idx - last_extract_hour) >= MIN_GAP_HOURS:
            peak_cell    = int(np.nanargmax(scores))
            peak_lat_idx = peak_cell // N_LON
            peak_lon_idx = peak_cell % N_LON

            current_patch = extract_patch(grid, peak_lat_idx, peak_lon_idx)

            ctx_list  = list(buffer)[:-1]
            ctx_grids = []
            for _ in range(CONTEXT_HOURS - len(ctx_list)):
                ctx_grids.append(np.zeros((PATCH_SIZE, PATCH_SIZE, N_FEATURES),
                                           dtype=np.float32))
            for _, g in ctx_list[-CONTEXT_HOURS:]:
                ctx_grids.append(extract_patch(g, peak_lat_idx, peak_lon_idx))
            context_stack = np.stack(ctx_grids, axis=0)

            # Write single patch directly to disk — NO augmentation
            np.save(os.path.join(TMP_DIR, f"c_{patch_index:06d}.npy"),
                    current_patch.transpose(2, 0, 1).astype(np.float32))
            np.save(os.path.join(TMP_DIR, f"ctx_{patch_index:06d}.npy"),
                    context_stack.transpose(0, 3, 1, 2).astype(np.float32))
            scores_list.append(peak_score)
            patch_index += 1
            n_extracted += 1
            last_extract_hour = hour_idx

    if (hour_idx + 1) % 2000 == 0:
        elapsed = time.time() - t0
        rate    = (hour_idx + 1) / elapsed
        eta     = (N_HOURS - hour_idx - 1) / rate
        print(f"  ... {hour_idx+1:,}/{N_HOURS} hrs | "
              f"flood hrs: {n_flood_seen} | "
              f"extracted: {n_extracted} | "
              f"{elapsed/60:.1f}min | ~{eta/60:.0f}min remaining")

elapsed = time.time() - t0
print(f"\n✓ Scan complete in {elapsed/60:.1f} min")
print(f"  Flood hours seen:  {n_flood_seen:,}")
print(f"  Patches extracted: {n_extracted:,}")
print(f"  Tmp files written: {patch_index * 2:,}")

# Save scores separately
np.save(os.path.join(OUTPUT_DIR, "patches_scores_tmp.npy"),
        np.array(scores_list, dtype=np.float32))
print(f"\n✓ Scores saved -> {OUTPUT_DIR}/patches_scores_tmp.npy")
print("Next: run wgan_stack_patches.py to stack tmp files into final arrays")

Source: merged_output_hourly_imerg/merged_data_normalised.csv (79.65 GB)
Features: 21, Patch: 64x64
Min gap between patches: 1h (no augmentation)

Single-pass scan — writing one patch per flood hour to disk...
  ... 2,000/21888 hrs | flood hrs: 1590 | extracted: 1590 | 1.0min | ~10min remaining
  ... 4,000/21888 hrs | flood hrs: 2936 | extracted: 2936 | 1.8min | ~8min remaining
  ... 6,000/21888 hrs | flood hrs: 4035 | extracted: 4035 | 2.6min | ~7min remaining
  ... 8,000/21888 hrs | flood hrs: 5282 | extracted: 5282 | 3.4min | ~6min remaining
  ... 10,000/21888 hrs | flood hrs: 6686 | extracted: 6686 | 4.3min | ~5min remaining
  ... 12,000/21888 hrs | flood hrs: 7521 | extracted: 7521 | 5.1min | ~4min remaining
  ... 14,000/21888 hrs | flood hrs: 8616 | extracted: 8616 | 5.8min | ~3min remaining
  ... 16,000/21888 hrs | flood hrs: 9804 | extracted: 9804 | 6.5min | ~2min remaining
  ... 18,000/21888 hrs | flood hrs: 11382 | extracted: 11382 | 7.3min | ~2min remaining
  ... 20,000/2188

In [13]:
import numpy as np
scores = np.full(19352, 95.0, dtype=np.float32)
np.save("wgan_patches/patches_scores.npy", scores)
print("✓ Scores set to 95.0 (flood threshold lower bound)")

✓ Scores set to 95.0 (flood threshold lower bound)


In [14]:
# WGAN STACK PATCHES v2 — memory-mapped stacking with progress bar
# ============================================================================
# Updated from v1:
#   - Works with any patch count (reads N dynamically from tmp files)
#   - Added tqdm progress bar for stacking visibility
#   - Compatible with v5 extractor output (no augmentation, 1h gap,
#     ~14,000 patches expected)
# ============================================================================
import os
import shutil
import numpy as np
from tqdm import tqdm

OUTPUT_DIR = "wgan_patches"
TMP_DIR    = os.path.join(OUTPUT_DIR, "tmp_patches")

# Count available patches
patch_files = sorted([f for f in os.listdir(TMP_DIR) if f.startswith("c_")])
N = len(patch_files)
print(f"Found {N} patch files in {TMP_DIR}")

if N == 0:
    print("⚠ No patch files found — run wgan_patch_extractor_v5.py first")
    raise SystemExit(1)

# Get shapes from first patch
sample_c   = np.load(os.path.join(TMP_DIR, "c_000000.npy"))
sample_ctx = np.load(os.path.join(TMP_DIR, "ctx_000000.npy"))
C_SHAPE   = sample_c.shape    # (21, 64, 64)
CTX_SHAPE = sample_ctx.shape  # (6, 21, 64, 64)

print(f"Current patch shape:  {C_SHAPE}")
print(f"Context patch shape:  {CTX_SHAPE}")
print(f"Estimated output sizes:")
print(f"  patches_current: {N * np.prod(C_SHAPE) * 4 / 1e9:.2f} GB")
print(f"  patches_context: {N * np.prod(CTX_SHAPE) * 4 / 1e9:.2f} GB\n")

# Create memory-mapped output arrays — written to disk, never loaded to RAM
out_c_path   = os.path.join(OUTPUT_DIR, "patches_current.npy")
out_ctx_path = os.path.join(OUTPUT_DIR, "patches_context.npy")
out_s_path   = os.path.join(OUTPUT_DIR, "patches_scores.npy")

print("Creating memory-mapped output files...")
mm_current = np.lib.format.open_memmap(
    out_c_path, mode="w+", dtype=np.float32, shape=(N,) + C_SHAPE)
mm_context = np.lib.format.open_memmap(
    out_ctx_path, mode="w+", dtype=np.float32, shape=(N,) + CTX_SHAPE)

# Stack in chunks with progress bar
CHUNK_SIZE = 500

with tqdm(total=N, desc="Stacking patches", unit="patch",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} patches "
                     "[{elapsed}<{remaining}, {rate_fmt}]") as pbar:
    for start in range(0, N, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, N)
        for i in range(start, end):
            idx = f"{i:06d}"
            mm_current[i] = np.load(os.path.join(TMP_DIR, f"c_{idx}.npy"))
            mm_context[i] = np.load(os.path.join(TMP_DIR, f"ctx_{idx}.npy"))
            pbar.update(1)
        mm_current.flush()
        mm_context.flush()

# Load scores
scores_file = os.path.join(OUTPUT_DIR, "patches_scores_tmp.npy")
if os.path.exists(scores_file):
    scores_arr = np.load(scores_file)
    print(f"✓ Scores loaded from {scores_file}")
else:
    scores_arr = np.full(N, 95.0, dtype=np.float32)
    print("⚠ Scores file not found — scores set to 95.0 (flood threshold)")

np.save(out_s_path, scores_arr)

print(f"\n✓ Stacking complete")
print(f"  patches_current: {out_c_path} "
      f"({os.path.getsize(out_c_path)/1e9:.2f} GB)")
print(f"  patches_context: {out_ctx_path} "
      f"({os.path.getsize(out_ctx_path)/1e9:.2f} GB)")
print(f"  patches_scores:  {out_s_path}")

# Clean up tmp files
response = input(f"\nDelete tmp_patches directory ({TMP_DIR})? (yes/no): ")
if response.strip().lower() == "yes":
    shutil.rmtree(TMP_DIR)
    print(f"✓ Deleted {TMP_DIR}")
else:
    print("Kept tmp files — delete manually when satisfied with output.")

print("\nNext steps:")
print("  1. Fill NaN in patches_current.npy and patches_context.npy")
print("  2. Pre-flatten patches_context.npy for training")
print("  3. Run wgan_train_v5.py")

Found 14002 patch files in wgan_patches/tmp_patches
Current patch shape:  (21, 64, 64)
Context patch shape:  (6, 21, 64, 64)
Estimated output sizes:
  patches_current: 4.82 GB
  patches_context: 28.91 GB

Creating memory-mapped output files...


Stacking patches: 100%|██████████| 14002/14002 patches [01:34<00:00, 147.76patch/s]

✓ Scores loaded from wgan_patches/patches_scores_tmp.npy

✓ Stacking complete
  patches_current: wgan_patches/patches_current.npy (4.82 GB)
  patches_context: wgan_patches/patches_context.npy (28.91 GB)
  patches_scores:  wgan_patches/patches_scores.npy


✓ Deleted wgan_patches/tmp_patches

Next steps:
  1. Fill NaN in patches_current.npy and patches_context.npy
  2. Pre-flatten patches_context.npy for training
  3. Run wgan_train_v5.py


In [ ]:
# WGAN MODEL v2 — U-NET GENERATOR + CRITIC (fixed channel dimensions)
# ============================================================================
# Fixes RuntimeError in dec4: UpBlock was adding in_ch + skip_ch in the
# conv layer but in_ch already included the skip channels from concatenation,
# causing a 1536 vs 1024 channel mismatch.
#
# Fix: UpBlock now takes (upsample_ch, skip_ch, out_ch) where upsample_ch
# is the channels BEFORE concatenation. The conv receives upsample_ch +
# skip_ch channels, which is correct.
# ============================================================================
import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    """Conv2d → BatchNorm → LeakyReLU"""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        # ConvBlock — change BatchNorm2d to InstanceNorm2d
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),  # ← changed
            nn.LeakyReLU(0.2, inplace=True),
        )
    
    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    """Upsample → concat skip → Conv2d → BatchNorm → ReLU
    Args:
        up_ch   : channels of the upsampled input (BEFORE concat with skip)
        skip_ch : channels of the skip connection
        out_ch  : output channels after conv
    """
    def __init__(self, up_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear",
                               align_corners=False)
        # UpBlock — same change
        self.block = nn.Sequential(
            nn.Conv2d(up_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),  # ← changed
            nn.ReLU(inplace=True),
        )
    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)  # (B, up_ch + skip_ch, H, W)
        return self.block(x)


class UNetGenerator(nn.Module):
    """
    Conditional U-Net generator.
    Input:
        context : (B, 6*21, 64, 64) — 6 conditioning hours stacked on channels
        noise   : (B, latent_dim)   — random noise vector
    Output:
        patch   : (B, n_features, 64, 64) — synthetic flood patch
    """
    def __init__(self, n_features=21, context_hours=6, latent_dim=128):
        super().__init__()
        in_ch = n_features * context_hours  # 126

        # Encoder
        self.enc1 = ConvBlock(in_ch, 64)           # (B, 64,  64, 64)
        self.enc2 = ConvBlock(64,  128, stride=2)   # (B, 128, 32, 32)
        self.enc3 = ConvBlock(128, 256, stride=2)   # (B, 256, 16, 16)
        self.enc4 = ConvBlock(256, 512, stride=2)   # (B, 512,  8,  8)

        # Bottleneck
        self.bottleneck = ConvBlock(512, 512, stride=2)  # (B, 512, 4, 4)
        self.noise_proj  = nn.Linear(latent_dim, 512 * 4 * 4)

        # Decoder — up_ch is channels BEFORE concat, skip_ch is skip channels
        self.dec4 = UpBlock(up_ch=512, skip_ch=512, out_ch=256)  # → (B, 256, 8,  8)
        self.dec3 = UpBlock(up_ch=256, skip_ch=256, out_ch=128)  # → (B, 128, 16, 16)
        self.dec2 = UpBlock(up_ch=128, skip_ch=128, out_ch=64)   # → (B, 64,  32, 32)
        self.dec1 = UpBlock(up_ch=64,  skip_ch=64,  out_ch=64)   # → (B, 64,  64, 64)

        # Output
        self.output = nn.Sequential(
            nn.Conv2d(64, n_features, 1),
            nn.Tanh(),
        )

    def forward(self, context, noise):
        e1 = self.enc1(context)       # (B, 64,  64, 64)
        e2 = self.enc2(e1)            # (B, 128, 32, 32)
        e3 = self.enc3(e2)            # (B, 256, 16, 16)
        e4 = self.enc4(e3)            # (B, 512,  8,  8)
        bn = self.bottleneck(e4)      # (B, 512,  4,  4)

        # Inject noise
        B = noise.shape[0]
        noise_map = self.noise_proj(noise).view(B, 512, 4, 4)
        bn = bn + noise_map

        d4 = self.dec4(bn, e4)        # (B, 256,  8,  8)
        d3 = self.dec3(d4, e3)        # (B, 128, 16, 16)
        d2 = self.dec2(d3, e2)        # (B, 64,  32, 32)
        d1 = self.dec1(d2, e1)        # (B, 64,  64, 64)
        return self.output(d1)        # (B, 21,  64, 64)


class Critic(nn.Module):
    """
    WGAN critic — no sigmoid, raw scalar Wasserstein score.
    Input:
        patch   : (B, 21,  64, 64)
        context : (B, 126, 64, 64)
    Concatenated → (B, 147, 64, 64) before processing.
    """
    def __init__(self, n_features=21, context_hours=6):
        super().__init__()
        in_ch = n_features + n_features * context_hours  # 147

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 64,  4, stride=2, padding=1),   # 64→32
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64,  128, 4, stride=2, padding=1),      # 32→16
            nn.InstanceNorm2d(128, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, stride=2, padding=1),      # 16→8
            nn.InstanceNorm2d(256, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, stride=2, padding=1),      # 8→4
            nn.InstanceNorm2d(512, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1,   4, stride=1, padding=0),      # 4→1
        )

    def forward(self, patch, context):
        x = torch.cat([patch, context], dim=1)  # (B, 147, 64, 64)
        return self.net(x).view(-1)             # (B,)


def gradient_penalty(critic, real, fake, context, device):
    """WGAN-GP gradient penalty."""
    B = real.shape[0]
    alpha = torch.rand(B, 1, 1, 1, device=device)
    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = critic(interpolated, context)
    grad = torch.autograd.grad(
        outputs=d_interp,
        inputs=interpolated,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
    )[0]
    grad_norm = grad.view(B, -1).norm(2, dim=1)
    return ((grad_norm - 1) ** 2).mean()


if __name__ == "__main__":
    device = torch.device("cpu")
    G = UNetGenerator(n_features=21, context_hours=6, latent_dim=128).to(device)
    C = Critic(n_features=21, context_hours=6).to(device)

    B = 2
    context = torch.randn(B, 6 * 21, 64, 64)
    noise   = torch.randn(B, 128)
    fake    = G(context, noise)
    score   = C(fake, context)

    assert fake.shape  == (B, 21, 64, 64), f"Wrong generator shape: {fake.shape}"
    assert score.shape == (B,),            f"Wrong critic shape: {score.shape}"

    G_params = sum(p.numel() for p in G.parameters())
    C_params = sum(p.numel() for p in C.parameters())
    print(f"Generator output shape: {fake.shape}")
    print(f"Critic output shape:    {score.shape}")
    print(f"Generator parameters:   {G_params:,}")
    print(f"Critic parameters:      {C_params:,}")
    print("✓ Architecture check passed")

Generator output shape: torch.Size([2, 21, 64, 64])
Critic output shape:    torch.Size([2])
Generator parameters:   8,212,565
Critic parameters:      2,913,985
✓ Architecture check passed


In [16]:
import numpy as np

# ── Fix patches_current.npy ──
patches = np.load("wgan_patches/patches_current.npy")
print(f"NaN before fill: {np.isnan(patches).sum():,}")
patches = np.nan_to_num(patches, nan=0.0)
print(f"NaN after fill:  {np.isnan(patches).sum():,}")
np.save("wgan_patches/patches_current.npy", patches.astype(np.float32))
print("✓ patches_current.npy cleaned\n")

# ── Fix patches_context_flat.npy (memmap — too large to load fully) ──
N = np.load("wgan_patches/patches_current.npy", mmap_mode='r').shape[0]
print(f"Total patches: {N:,}")

ctx = np.memmap("wgan_patches/patches_context_flat.npy",
                dtype=np.float32, mode='r+',
                shape=(N, 126, 64, 64))
nan_count = int(np.isnan(ctx).sum())
print(f"NaN in context: {nan_count:,}")
if nan_count > 0:
    ctx[np.isnan(ctx)] = 0.0
    print("✓ NaN filled in context patches")
else:
    print("✓ Context patches already clean")

NaN before fill: 33,110,906
NaN after fill:  0
✓ patches_current.npy cleaned

Total patches: 14,002
NaN in context: 198,629,589
✓ NaN filled in context patches


In [1]:
# WGAN-GP TRAINING LOOP v5
# ============================================================================
# Changes from v4:
#   1. BATCH_SIZE increased from 4 to 8 for more stable gradients
#   2. N_EPOCHS increased from 20 to 40
#   3. LR reduced from 1e-4 to 5e-5 for more stable convergence
#   4. SAVE_EVERY and SAMPLE_EVERY set to 10
#   5. NaN check on first sample batch before training starts
#   6. Gradient clipping added to prevent NaN recurrence
# ============================================================================
import os
import time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from wgan_model_v2 import UNetGenerator, Critic, gradient_penalty

# ── Config ──
PATCH_DIR      = "wgan_patches"
CHECKPOINT_DIR = "wgan_checkpoints"
SAMPLE_DIR     = "wgan_samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

N_FEATURES     = 21
CONTEXT_HOURS  = 6
LATENT_DIM     = 128
N_CRITIC       = 5
LAMBDA_GP      = 10
SUBSAMPLE_STEP = 4
BATCH_SIZE     = 8      # increased from 4
N_EPOCHS       = 40     # increased from 20
SAVE_EVERY     = 10
SAMPLE_EVERY   = 10
LR             = 5e-5   # reduced from 1e-4
MAX_GRAD_NORM  = 1.0    # gradient clipping to prevent NaN recurrence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Fix scores if all zeros
scores_path = os.path.join(PATCH_DIR, "patches_scores.npy")
scores = np.load(scores_path)
if scores.max() == 0:
    scores[:] = 95.0
    np.save(scores_path, scores)
    print("✓ Scores fixed — set to 95.0")

# ── Pre-flatten context if needed ──
flat_path = os.path.join(PATCH_DIR, "patches_context_flat.npy")
if not os.path.exists(flat_path):
    print("\nPre-flattening context (6,21,64,64) → (126,64,64)...")
    ctx_src = np.load(os.path.join(PATCH_DIR, "patches_context.npy"),
                      mmap_mode='r')
    N = ctx_src.shape[0]
    ctx_flat = np.memmap(flat_path, dtype=np.float32, mode='w+',
                         shape=(N, 126, 64, 64))
    CHUNK = 100
    for i in tqdm(range(0, N, CHUNK), desc="Flattening context"):
        end = min(i + CHUNK, N)
        ctx_flat[i:end] = ctx_src[i:end].reshape(end - i, 126, 64, 64)
    ctx_flat.flush()
    print(f"✓ Saved {flat_path} ({os.path.getsize(flat_path)/1e9:.1f} GB)\n")
else:
    print(f"✓ Found existing {flat_path} — skipping pre-flatten")


# ── Memory-mapped Dataset with subsampling ──
class FloodPatchDataset(Dataset):
    def __init__(self, patch_dir, subsample_step=1):
        current_all = np.load(
            os.path.join(patch_dir, "patches_current.npy"), mmap_mode='r')
        N = len(current_all)
        context_all = np.memmap(flat_path, dtype=np.float32, mode='r',
                                shape=(N, 126, 64, 64))
        indices = np.arange(0, N, subsample_step)
        self.current = current_all[indices]
        self.context = context_all[indices]
        print(f"Dataset: {len(self.current)} patches "
              f"(subsampled 1/{subsample_step} from {N})")
        print(f"  current: {self.current.shape}")
        print(f"  context: {self.context.shape}")

    def __len__(self):
        return len(self.current)

    def __getitem__(self, idx):
        current = torch.from_numpy(self.current[idx].copy()).float()
        context = torch.from_numpy(self.context[idx].copy()).float()
        return current, context


# ── Init ──
dataset    = FloodPatchDataset(PATCH_DIR, subsample_step=SUBSAMPLE_STEP)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, drop_last=True, pin_memory=False)

# NaN check on first batch before training
sample_c, sample_ctx = dataset[0]
assert not sample_c.isnan().any(),   "⚠ NaN in current patches — re-run NaN fill"
assert not sample_ctx.isnan().any(), "⚠ NaN in context patches — re-run NaN fill"
print("✓ NaN check passed — patches are clean")

G = UNetGenerator(N_FEATURES, CONTEXT_HOURS, LATENT_DIM).to(device)
C = Critic(N_FEATURES, CONTEXT_HOURS).to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.0, 0.9))
opt_C = torch.optim.Adam(C.parameters(), lr=LR, betas=(0.0, 0.9))

# Resume from checkpoint if available
start_epoch = 0
latest_ckpt = os.path.join(CHECKPOINT_DIR, "latest.pt")
if os.path.exists(latest_ckpt):
    ckpt = torch.load(latest_ckpt, map_location=device)
    G.load_state_dict(ckpt["G"])
    C.load_state_dict(ckpt["C"])
    opt_G.load_state_dict(ckpt["opt_G"])
    opt_C.load_state_dict(ckpt["opt_C"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from epoch {start_epoch}")

# Fixed samples for consistent monitoring
fixed_noise   = torch.randn(2, LATENT_DIM, device=device)
fixed_context = next(iter(dataloader))[1][:2].to(device)

# ── Training loop ──
n_batches_per_epoch = len(dataloader)
print(f"\nTraining cWGAN-GP:")
print(f"  Epochs:        {N_EPOCHS}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Batches/epoch: {n_batches_per_epoch}")
print(f"  LR:            {LR}")
print(f"  Critic steps:  {N_CRITIC}x per generator step\n")

loss_log = []
t_start  = time.time()

with tqdm(total=N_EPOCHS - start_epoch, desc="Overall", unit="epoch",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} epochs "
                     "[{elapsed}<{remaining}] {postfix}") as epoch_bar:

    for epoch in range(start_epoch, N_EPOCHS):
        G.train(); C.train()
        epoch_c_loss = 0.0
        epoch_g_loss = 0.0
        n_batches    = 0
        nan_batches  = 0

        with tqdm(total=n_batches_per_epoch,
                  desc=f"  Epoch {epoch+1:2d}/{N_EPOCHS}",
                  unit="batch", leave=False,
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} "
                             "[{elapsed}<{remaining}, {rate_fmt}] "
                             "{postfix}") as batch_bar:

            for real, context in dataloader:
                real    = real.to(device)
                context = context.to(device)
                B       = real.shape[0]

                # ── Critic steps ──
                for _ in range(N_CRITIC):
                    noise = torch.randn(B, LATENT_DIM, device=device)
                    with torch.no_grad():
                        fake = G(context, noise)
                    C.zero_grad()
                    c_real = C(real, context).mean()
                    c_fake = C(fake.detach(), context).mean()
                    gp     = gradient_penalty(C, real, fake.detach(),
                                              context, device)
                    c_loss = c_fake - c_real + LAMBDA_GP * gp
                    c_loss.backward()
                    torch.nn.utils.clip_grad_norm_(C.parameters(),
                                                   MAX_GRAD_NORM)
                    opt_C.step()

                # ── Generator step ──
                noise  = torch.randn(B, LATENT_DIM, device=device)
                fake   = G(context, noise)
                G.zero_grad()
                g_loss = -C(fake, context).mean()
                g_loss.backward()
                torch.nn.utils.clip_grad_norm_(G.parameters(),
                                               MAX_GRAD_NORM)
                opt_G.step()

                # Track NaN occurrences
                if torch.isnan(c_loss) or torch.isnan(g_loss):
                    nan_batches += 1
                else:
                    epoch_c_loss += c_loss.item()
                    epoch_g_loss += g_loss.item()
                    n_batches    += 1

                batch_bar.set_postfix(
                    C=f"{c_loss.item():+.3f}" if not torch.isnan(c_loss)
                      else "NaN",
                    G=f"{g_loss.item():+.3f}" if not torch.isnan(g_loss)
                      else "NaN",
                    refresh=False)
                batch_bar.update(1)

        avg_c   = epoch_c_loss / max(n_batches, 1)
        avg_g   = epoch_g_loss / max(n_batches, 1)
        elapsed = (time.time() - t_start) / 60
        eta     = (elapsed / (epoch - start_epoch + 1) *
                   (N_EPOCHS - epoch - 1)) if epoch > start_epoch else 0

        loss_log.append({"epoch": epoch + 1, "c_loss": avg_c,
                         "g_loss": avg_g, "nan_batches": nan_batches})

        postfix = dict(C=f"{avg_c:+.4f}", G=f"{avg_g:+.4f}",
                       elapsed=f"{elapsed:.0f}min", ETA=f"{eta:.0f}min")
        if nan_batches > 0:
            postfix["NaN"] = str(nan_batches)
        epoch_bar.set_postfix(**postfix)
        epoch_bar.update(1)

        # Checkpoint
        if (epoch + 1) % SAVE_EVERY == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR,
                                     f"epoch_{epoch+1:04d}.pt")
            state = {
                "epoch": epoch,
                "G": G.state_dict(), "C": C.state_dict(),
                "opt_G": opt_G.state_dict(), "opt_C": opt_C.state_dict(),
                "c_loss": avg_c, "g_loss": avg_g,
            }
            torch.save(state, ckpt_path)
            torch.save(state, latest_ckpt)
            tqdm.write(f"  ✓ Checkpoint -> {ckpt_path}")

        # Sample patches
        if (epoch + 1) % SAMPLE_EVERY == 0:
            G.eval()
            with torch.no_grad():
                samples = G(fixed_context, fixed_noise).cpu().numpy()
            sample_path = os.path.join(SAMPLE_DIR,
                                       f"samples_epoch_{epoch+1:04d}.npy")
            np.save(sample_path, samples)
            tqdm.write(f"  ✓ Samples -> {sample_path}")

import pandas as pd
pd.DataFrame(loss_log).to_csv(
    os.path.join(CHECKPOINT_DIR, "loss_log.csv"), index=False)
tqdm.write(f"\n✓ Training complete in {(time.time()-t_start)/60:.0f} min")
tqdm.write(f"  Checkpoints: {CHECKPOINT_DIR}/")
tqdm.write(f"  Loss log:    {CHECKPOINT_DIR}/loss_log.csv")
tqdm.write(f"  Next: run wgan_generate.py to generate synthetic patches")

Device: cpu
✓ Found existing wgan_patches/patches_context_flat.npy — skipping pre-flatten
Dataset: 3501 patches (subsampled 1/4 from 14002)
  current: (3501, 21, 64, 64)
  context: (3501, 126, 64, 64)
✓ NaN check passed — patches are clean

Training cWGAN-GP:
  Epochs:        40
  Batch size:    8
  Batches/epoch: 437
  LR:            5e-05
  Critic steps:  5x per generator step



Overall:  25%|██▌       | 10/40 epochs [5:25:26<11:39:43] , C=-6995.9099, ETA=976min, G=+8199.4067, elapsed=325min

  ✓ Checkpoint -> wgan_checkpoints/epoch_0010.pt
  ✓ Samples -> wgan_samples/samples_epoch_0010.npy


Overall:  50%|█████     | 20/40 epochs [8:41:14<6:23:16] , C=+46336.0914, ETA=521min, G=+22523.6970, elapsed=521min 

  ✓ Checkpoint -> wgan_checkpoints/epoch_0020.pt
  ✓ Samples -> wgan_samples/samples_epoch_0020.npy


Overall:  75%|███████▌  | 30/40 epochs [11:41:05<3:01:59] , C=+387932.4726, ETA=234min, G=+44513.3640, elapsed=701min

  ✓ Checkpoint -> wgan_checkpoints/epoch_0030.pt
  ✓ Samples -> wgan_samples/samples_epoch_0030.npy


Overall: 100%|██████████| 40/40 epochs [14:50:32<00:00] , C=+2004619.5619, ETA=0min, G=+76181.7595, elapsed=891min    


  ✓ Checkpoint -> wgan_checkpoints/epoch_0040.pt
  ✓ Samples -> wgan_samples/samples_epoch_0040.npy

✓ Training complete in 891 min
  Checkpoints: wgan_checkpoints/
  Loss log:    wgan_checkpoints/loss_log.csv
  Next: run wgan_generate.py to generate synthetic patches


In [ ]:
# WGAN SYNTHETIC PATCH GENERATOR v2 — WITH PROGRESS BAR
# ============================================================================
# Updated from v1:
#   - Context shape derived dynamically from patches_current.npy
#     rather than hardcoded (fixes ValueError: mmap length > file size)
#   - Compatible with v5 extractor output (~14,000 patches, no augmentation)
# ============================================================================
import os
import numpy as np
import torch
from tqdm import tqdm
from wgan_model_v2 import UNetGenerator

PATCH_DIR      = "wgan_patches"
CHECKPOINT_DIR = "wgan_checkpoints"
N_FEATURES     = 21
CONTEXT_HOURS  = 6
LATENT_DIM     = 128
N_SYNTHETIC    = 6000
BATCH_SIZE     = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load checkpoint
ckpt_path = os.path.join(CHECKPOINT_DIR, "latest.pt")
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f"No checkpoint at {ckpt_path}. Run wgan_train_v5.py first.")

ckpt = torch.load(ckpt_path, map_location=device)
G = UNetGenerator(N_FEATURES, CONTEXT_HOURS, LATENT_DIM).to(device)
G.load_state_dict(ckpt["G"])
G.eval()
print(f"Loaded generator from epoch {ckpt['epoch']+1}")

# Derive patch count dynamically — never hardcode this
N_REAL = np.load(os.path.join(PATCH_DIR, "patches_current.npy"),
                 mmap_mode='r').shape[0]
print(f"Real patches available: {N_REAL}")

# Load pre-flattened context with correct shape
ctx_flat = np.memmap(os.path.join(PATCH_DIR, "patches_context_flat.npy"),
                     dtype=np.float32, mode='r',
                     shape=(N_REAL, 126, 64, 64))
print(f"Context shape: {ctx_flat.shape}")
print(f"Generating {N_SYNTHETIC} synthetic patches (batch_size={BATCH_SIZE})...\n")

n_batches        = int(np.ceil(N_SYNTHETIC / BATCH_SIZE))
synthetic_patches = []
n_generated      = 0

with torch.no_grad():
    with tqdm(total=N_SYNTHETIC, desc="Generating patches", unit="patch",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} patches "
                         "[{elapsed}<{remaining}, {rate_fmt}] {postfix}") as pbar:
        while n_generated < N_SYNTHETIC:
            batch_size = min(BATCH_SIZE, N_SYNTHETIC - n_generated)
            idx     = np.random.randint(0, N_REAL, size=batch_size)
            context = torch.from_numpy(ctx_flat[idx].copy()).to(device)
            noise   = torch.randn(batch_size, LATENT_DIM, device=device)
            fake    = G(context, noise).cpu().numpy()
            synthetic_patches.append(fake)
            n_generated += batch_size
            pbar.update(batch_size)
            pbar.set_postfix(
                batches=f"{len(synthetic_patches)}/{n_batches}",
                refresh=False)

synthetic_patches = np.concatenate(synthetic_patches, axis=0)[:N_SYNTHETIC]
print(f"\n✓ Generated {len(synthetic_patches)} synthetic patches")
print(f"  Shape:       {synthetic_patches.shape}")
print(f"  Value range: [{synthetic_patches.min():.4f}, "
      f"{synthetic_patches.max():.4f}]")

# Quality check
print("\nQuality check (synthetic vs real statistics):")
real_patches = np.load(os.path.join(PATCH_DIR, "patches_current.npy"),
                       mmap_mode='r')
print(f"  Real      — mean: {real_patches.mean():.4f},  "
      f"std: {real_patches.std():.4f}")
print(f"  Synthetic — mean: {synthetic_patches.mean():.4f},  "
      f"std: {synthetic_patches.std():.4f}")

mean_diff = abs(float(real_patches.mean()) - synthetic_patches.mean())
std_diff  = abs(float(real_patches.std())  - synthetic_patches.std())
if mean_diff < 0.5 and std_diff < 0.5:
    print("  ✓ Statistics look reasonable — synthetic data is plausible")
else:
    print(f"  ⚠ mean diff={mean_diff:.4f}, std diff={std_diff:.4f} — "
          "consider more training epochs before generating")

# Check for NaN
nan_count = int(np.isnan(synthetic_patches).sum())
if nan_count > 0:
    print(f"  ⚠ {nan_count:,} NaN values in synthetic patches — "
          "training may not have converged")
else:
    print("  ✓ No NaN values in synthetic patches")

# Save
out_path = os.path.join(PATCH_DIR, "synthetic_current.npy")
np.save(out_path, synthetic_patches.astype(np.float32))
print(f"\n✓ Saved -> {out_path} ({synthetic_patches.nbytes/1e6:.0f} MB)")
print("Next: use synthetic_current.npy during ConvLSTM/3D-CNN training.")

Device: cpu
Loaded generator from epoch 40


ValueError: mmap length is greater than file size

In [4]:
import pandas as pd
log = pd.read_csv("wgan_checkpoints/loss_log.csv")
print(log.to_string())

    epoch        c_loss         g_loss  nan_batches
0       1 -7.986587e+02     400.676826            0
1       2 -2.754562e+03    1379.900055            0
2       3 -5.099134e+03    2558.699964            0
3       4 -7.473998e+03    3784.223340            0
4       5 -8.170082e+03    4936.594102            0
5       6 -6.472050e+03    6189.113228            0
6       7 -6.159170e+03    7571.858476            0
7       8 -4.712388e+03    9185.132701            0
8       9 -2.016315e+03   10826.992207            0
9      10  9.250464e+03   12779.083802            0
10     11  1.314070e+04   14570.102669            0
11     12  3.685323e+04   16960.442087            0
12     13  2.757610e+04   19349.237563            0
13     14  4.918045e+04   21787.772930            0
14     15  7.835137e+04   24230.746333            0
15     16  7.331992e+04   27017.242234            0
16     17  1.482075e+05   30004.402143            0
17     18  2.488803e+05   33194.280131            0
18     19  3

In [1]:
import numpy as np

patches = np.load("wgan_patches/patches_current.npy")
print(f"NaN before fill: {np.isnan(patches).sum():,}")

# Fill NaN with 0 (appropriate for z-score normalised data —
# 0 represents the mean value, a neutral fill for missing cells)
patches = np.nan_to_num(patches, nan=0.0)
print(f"NaN after fill:  {np.isnan(patches).sum():,}")

np.save("wgan_patches/patches_current.npy", patches.astype(np.float32))
print("✓ Saved clean patches")

NaN before fill: 0
NaN after fill:  0
✓ Saved clean patches


In [2]:
# For the flat context file — needs memmap approach given its size
import numpy as np
import os

ctx = np.memmap("wgan_patches/patches_context_flat.npy",
                dtype=np.float32, mode='r+',
                shape=(19352, 126, 64, 64))

nan_count = np.isnan(ctx).sum()
print(f"NaN in context: {nan_count:,}")
if nan_count > 0:
    ctx[np.isnan(ctx)] = 0.0
    print("✓ NaN filled in context patches (in-place memmap edit)")

NaN in context: 0
